[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pandera-certified/notebooks/day-02-check-builtins-custom.ipynb#scrollTo=aa11bb22)

---
# Day 2 · Built-in Checks and Custom Lambda Checks
**certified-journeys / pandera-certified** · Day 2 · Value-Level Validation

> **Goal for today:** Apply built-in `Check` methods to numeric, categorical, and string columns, then write both element-wise and vectorized custom checks — and combine multiple checks on a single column.


In [ ]:
%pip install -q pandera


## Step 1 · Check reference — element-wise vs vectorized

Every Pandera `Check` is one of two execution modes:

| Mode | Input to function | When to use |
|---|---|---|
| **Vectorized** (default) | Entire `pd.Series` | Aggregate checks, `.str` accessor, set membership |
| **Element-wise** (`element_wise=True`) | Single scalar value | Per-row arithmetic, regex on one string |

Built-in checks are always **vectorized** internally (Pandera handles dispatch).
Custom `Check(fn)` defaults to vectorized — pass `element_wise=True` for scalar logic.

Full Check API: https://pandera.readthedocs.io/en/stable/reference/generated/pandera.api.checks.Check.html


In [ ]:
import pandera as pa
import pandas as pd
import numpy as np

# Quick demo: same check written both ways
# Vectorized: receives the whole Series
check_vectorized = pa.Check(lambda s: (s > 0).all())

# Element-wise: receives one scalar at a time
check_elementwise = pa.Check(lambda x: x > 0, element_wise=True)

# Both express the same constraint — "all prices must be positive"
schema_prices = pa.DataFrameSchema({
    "price_v": pa.Column(float, check_vectorized),
    "price_e": pa.Column(float, check_elementwise),
})

df_demo = pd.DataFrame({
    "price_v": [10.0, 20.0, 30.0],
    "price_e": [10.0, 20.0, 30.0],
})

result = schema_prices.validate(df_demo)
print("Both modes pass:", result.shape)
print("\nVectorized: function receives Series, returns bool")
print("Element-wise: function receives scalar, returns bool")


### What just happened?

- **Vectorized checks are faster** — Pandas operates on the entire Series without a Python loop.
- **Element-wise checks are easier to reason about** — the function takes a scalar and returns `True`/`False`.
- For built-in checks like `Check.greater_than(0)`, Pandera handles the vectorized implementation for you — you don't need to write `.all()`.
- Use vectorized for anything involving `.str`, aggregations, or set operations; use element-wise for per-row arithmetic or complex conditionals.


## Step 2 · Numeric built-ins — `greater_than` and `less_than_or_equal_to`

Pandera ships a rich set of numeric checks. The most common:

```python
pa.Check.greater_than(n)            # value > n
pa.Check.greater_than_or_equal_to(n) # value >= n
pa.Check.less_than(n)               # value < n
pa.Check.less_than_or_equal_to(n)   # value <= n
pa.Check.in_range(min_val, max_val) # min_val <= value <= max_val
```

Multiple checks on one column are passed as a **list**.


In [ ]:
# Products table: price > 0, age <= 120
schema_product = pa.DataFrameSchema({
    "product": pa.Column(str),
    "price":   pa.Column(float, pa.Check.greater_than(0)),          # strictly positive
    "age":     pa.Column(int,   pa.Check.less_than_or_equal_to(120)), # plausible human age
})

df_products = pd.DataFrame({
    "product": ["Widget", "Gadget", "Doohickey"],
    "price":   [9.99, 49.95, 0.50],
    "age":     [25, 110, 42],
})

validated = schema_product.validate(df_products)
print("Valid product table:")
print(validated)

# Introduce a bad row: price = -1 and age = 150
df_bad = pd.DataFrame({
    "product": ["Widget", "Bad Item"],
    "price":   [9.99, -1.00],      # negative price
    "age":     [25, 150],           # age > 120
})

try:
    schema_product.validate(df_bad, lazy=True)  # collect all failures
except pa.errors.SchemaErrors as e:
    print("\nAll failure cases:")
    print(e.failure_cases[["schema_context", "column", "check", "failure_case"]])


### What just happened?

- **`lazy=True`** runs every check before raising — the exception is `SchemaErrors` (plural) with all failures, not just the first.
- `failure_cases` includes a `column` and `check` column so you can pinpoint exactly which rule fired.
- **`Check.greater_than(0)` vs `Check.greater_than_or_equal_to(0)`:** decide based on your domain — a price of 0.00 may be valid for free trials.
- Numeric built-ins are the most concise way to express range constraints; prefer them over `Check(lambda x: x > 0, element_wise=True)` for readability.


## Step 3 · Categorical check — `Check.isin`

`Check.isin(allowed_values)` validates that every value in a column belongs
to a fixed set. This is the canonical way to validate **enum-like** columns
such as status codes, country codes, or lifecycle states.

```python
pa.Check.isin(['active', 'inactive', 'pending'])
```

There is also `Check.notin(forbidden_values)` for the inverse.


In [ ]:
# Orders table where status must be one of three known values
VALID_STATUSES = ["active", "inactive", "pending"]

schema_orders = pa.DataFrameSchema({
    "order_id": pa.Column(int),
    "amount":   pa.Column(float, pa.Check.greater_than(0)),
    "status":   pa.Column(str, pa.Check.isin(VALID_STATUSES)),
})

df_orders = pd.DataFrame({
    "order_id": [101, 102, 103],
    "amount":   [25.0, 100.0, 8.5],
    "status":   ["active", "pending", "inactive"],
})

validated = schema_orders.validate(df_orders)
print("Valid orders:", validated.shape)

# Introduce an unexpected status
df_bad_status = df_orders.copy()
df_bad_status.loc[1, "status"] = "archived"   # not in VALID_STATUSES

try:
    schema_orders.validate(df_bad_status)
except pa.errors.SchemaError as e:
    print("\nInvalid status caught:")
    print(e.failure_cases)


### What just happened?

- **`Check.isin` is the safest way to validate enums** — it catches typos (`"Inactive"` vs `"inactive"`) and unexpected values from upstream system changes.
- Define `VALID_STATUSES` as a constant so both the schema and your application logic share the same source of truth.
- **Case-sensitivity matters:** `"Active"` would fail `isin(["active", ...])` — decide on and enforce a canonical casing in your data contract.
- Combine with `nullable=False` (the default) to also reject `None` / `NaN` in the status column.


## Step 4 · String checks — `Check.str_matches`

`Check.str_matches(pattern)` validates that every value in a column matches
a regular expression. It uses `pd.Series.str.match()` under the hood (anchored
at the start — use `Check.str_contains` for substring matching).

Other useful string checks:
```python
pa.Check.str_contains(r'@')          # substring or regex
pa.Check.str_startswith('prefix_')   # literal prefix
pa.Check.str_endswith('.com')        # literal suffix
pa.Check.str_length(min_value=1, max_value=100)
```


In [ ]:
# Validate an email column with a simple regex
EMAIL_PATTERN = r'^\w+@\w+\.\w+$'   # basic: word@word.word

schema_users = pa.DataFrameSchema({
    "user_id": pa.Column(int),
    "email":   pa.Column(str, pa.Check.str_matches(EMAIL_PATTERN)),
})

df_users = pd.DataFrame({
    "user_id": [1, 2, 3],
    "email":   ["alice@example.com", "bob@work.org", "carol@mail.net"],
})

validated = schema_users.validate(df_users)
print("Valid email rows:")
print(validated)

# Bad emails that should fail
df_bad_emails = pd.DataFrame({
    "user_id": [4, 5],
    "email":   ["not-an-email", "missing@dot"],  # no TLD, no dot
})

try:
    schema_users.validate(df_bad_emails)
except pa.errors.SchemaError as e:
    print("\nBad emails caught:")
    print(e.failure_cases)


### What just happened?

- **`Check.str_matches` anchors at the start** — the pattern `r'^\w+@\w+\.\w+$'` must match the full string.
- `"missing@dot"` fails because `.\w+$` (the TLD part) is missing after the dot.
- For complex email validation, `Check.str_matches` with a real RFC 5322 regex is more robust than a custom check.
- **`Check.str_contains` vs `Check.str_matches`:** `str_contains` is a substring search; `str_matches` requires the pattern to match from position 0.


## Step 5 · Custom element-wise check

When no built-in covers your rule, write a lambda. Pass `element_wise=True`
when your function operates on a **single scalar value** rather than a Series.

```python
pa.Check(lambda x: x % 2 == 0, element_wise=True, error="must be even")
```

The optional `error` parameter customises the message in `SchemaError`.


In [ ]:
# Business rule: batch sizes must be even (pairs of items)
schema_batches = pa.DataFrameSchema({
    "batch_id":   pa.Column(str),
    "batch_size": pa.Column(
        int,
        [
            pa.Check.greater_than(0),                               # must be positive
            pa.Check(                                               # must be even
                lambda x: x % 2 == 0,
                element_wise=True,
                error="batch_size must be even",
            ),
        ]
    ),
})

df_batches = pd.DataFrame({
    "batch_id":   ["B001", "B002", "B003"],
    "batch_size": [10, 4, 20],              # all even, all positive
})

validated = schema_batches.validate(df_batches)
print("Valid batches:")
print(validated)

# Odd batch size should fail
df_odd = pd.DataFrame({
    "batch_id":   ["B004"],
    "batch_size": [7],                      # 7 is odd
})

try:
    schema_batches.validate(df_odd)
except pa.errors.SchemaError as e:
    print("\nOdd batch caught:")
    print(e)


### What just happened?

- **Multiple checks in a list** — Pandera runs them in order; the first failure raises immediately (unless `lazy=True`).
- `element_wise=True` means the lambda receives `7` (the integer), not the Series — makes scalar arithmetic natural.
- The `error` parameter appears in `SchemaError` messages — always use it for custom checks to avoid generic error messages.
- Checks compose: `greater_than(0)` AND the even check together express the full constraint `batch_size > 0 and batch_size % 2 == 0`.


## Step 6 · Custom vectorized check — Series-level logic

When your rule involves the **whole column at once** — length aggregates, `.str`
accessor, or cross-element constraints — write a vectorized check. The function
receives the `pd.Series` and must return a **boolean Series** (one `True`/`False`
per row).

```python
pa.Check(lambda s: s.str.len() <= 50)   # vectorized, returns bool Series
```


In [ ]:
# Product descriptions must be non-empty and max 50 chars
schema_catalog = pa.DataFrameSchema({
    "product_id":  pa.Column(int),
    "description": pa.Column(
        str,
        [
            # Vectorized: operates on the whole Series via .str accessor
            pa.Check(
                lambda s: s.str.len() >= 1,
                error="description cannot be empty",
            ),
            pa.Check(
                lambda s: s.str.len() <= 50,
                error="description exceeds 50 characters",
            ),
        ]
    ),
})

df_catalog = pd.DataFrame({
    "product_id":  [1, 2, 3],
    "description": ["Blue widget", "Premium gadget with extra grip", "Thing"],
})

validated = schema_catalog.validate(df_catalog)
print("Valid catalog:")
print(validated)

# Overly long description
df_long = pd.DataFrame({
    "product_id":  [4],
    "description": ["A" * 55],   # 55 chars — exceeds limit
})

try:
    schema_catalog.validate(df_long)
except pa.errors.SchemaError as e:
    print("\nLong description caught:")
    print(e.failure_cases)


### What just happened?

- **Vectorized custom checks return a bool Series** — each element is `True` (passes) or `False` (fails). Pandera converts the `False` rows into `failure_cases`.
- `.str.len()` is a natural Pandas Series operation — vectorized checks let you use the full Pandas API.
- **Combine built-ins and custom checks freely** in the same list — they all run as part of the same column schema.
- For string length specifically, `pa.Check.str_length(min_value=1, max_value=50)` is an equivalent built-in — use whichever is clearer.


In [ ]:
# Challenge: Build a schema for a transactions table with these rules:
#
#   - transaction_id: int, must be positive
#   - amount: float, must be > 0 and < 10000 (fraud threshold)
#   - status: str, must be one of ['completed', 'pending', 'failed']
#   - note: str, nullable, max 100 chars (vectorized custom check)
#   - ref_code: str, must match pattern r'^TXN-\d{6}$' (6-digit code)
#
# Then validate the following DataFrame and catch any failures.

df_transactions = pd.DataFrame({
    "transaction_id": [1001, 1002, 1003],
    "amount":         [250.0, 9999.99, 10.0],
    "status":         ["completed", "pending", "failed"],
    "note":           ["routine", None, "retry attempt"],
    "ref_code":       ["TXN-000001", "TXN-999999", "TXN-123456"],
})

# Your schema here:
# schema_transactions = pa.DataFrameSchema({
#     ...
# })

# Validate and print results or failure_cases


---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| Element-wise check | `element_wise=True` — lambda receives scalar, returns `bool` |
| Vectorized check | Default — lambda receives `pd.Series`, returns `bool Series` |
| `Check.greater_than(n)` | Value must be strictly > n; `_or_equal_to` variant for >= |
| `Check.isin(values)` | Column must only contain values from the given list |
| `Check.str_matches(r'...')` | Column must match regex (anchored at start) |
| Multiple checks | Pass a list `[check1, check2]` — all must pass |
| `lazy=True` | Collect all failures before raising (`SchemaErrors` plural) |

> **Tip:** Always name your custom checks with the `error=` parameter. Generic errors like "series did not pass" are useless in production logs.

---
## What's next
**Day 3** → `SchemaModel` — the class-based API with type annotations and `pa.Field()`, for schemas that are easier to reuse, subclass, and document.

Mark Day 2 complete in your [tracker](../index.html).
